---
# DNN : `Engie`
---

## Packages

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import joblib

## Configuration machine

Pour connaître le nombre de coeurs de sa machine :

In [ ]:
n_cores = joblib.cpu_count(only_physical_cores=True)
print(f'Nombre de coeurs (physiques): {n_cores}')

---
## 1. Données
---

### 1.1. Importation des données

In [ ]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google colab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google colab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

In [ ]:
X = pd.read_csv(directory + 'engie_X.csv',
                header    = 0,
                index_col = None,
                sep       = ';',
                decimal   = '.')

y = pd.read_csv(directory + 'engie_Y.csv',
                header    = 0,
                index_col = None,
                sep       = ';',
                decimal   = '.')

In [ ]:
X.info()

In [ ]:
X.head()

In [ ]:
y.info()

In [ ]:
y.head()

### 1.2. Analyse des données manquantes

On concatène les données pour éviter les erreurs de traitement :

In [ ]:
data = pd.merge(X, y, on='ID', how='inner')
print('Dimensions du jeu de données :', data.shape)

In [ ]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

On constate que les variables `Grid_voltage`, `Grid_voltage_min`, `Grid_voltage_max` et `Grid_voltage_std` présentent beaucoup de données manquantes (environ 16%) ; on analyse ces variables pour décider de leur retrait ou de leur imputation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data['Grid_voltage'], data['TARGET'])
ax.grid()
ax.set_xlabel('Grid_voltage')
ax.set_ylabel('TARGET')
plt.show()

Cette analyse nous incite à retirer ces variables dans une première approche :

In [ ]:
to_remove = data.columns[data.columns.str.startswith('Grid_voltage')]
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

### 1.3. Restriction du champs d'étude

On peut remarquer que les couples de variables `Absolute_wind_direction`-`Absolute_wind_direction_c` et `Nacelle_angle`-`Nacelle_angle_c` sont quasiment identiques (seules des valeurs manquantes les séparent), on en supprime une sur deux (celle ne présentant pas de données manquantes par hypothèse, il faudrait vérifier la pertinence de cette dernière auprès du "métier") :

In [ ]:
different_rows = data[['Absolute_wind_direction', 'Absolute_wind_direction_c']][data['Absolute_wind_direction'] != data['Absolute_wind_direction_c']]
print('Nombre de lignes différentes :', different_rows.shape[0])
different_rows.head()

In [ ]:
different_rows = data[['Nacelle_angle','Nacelle_angle_c']][data['Nacelle_angle'] != data['Nacelle_angle_c']]
print('Nombre de lignes différentes :', different_rows.shape[0])
different_rows.head()

In [ ]:
to_remove = ['Absolute_wind_direction_c','Nacelle_angle_c']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

On retire les observations pour lesquelles il subsiste des données manquantes :

In [ ]:
data.dropna(inplace=True)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
data['MAC_CODE'].value_counts()

On considère les cosinus et sinus des variables angulaires :

In [ ]:
data['Nacelle_angle_cos'] = np.cos(np.cos(data['Nacelle_angle']))
data['Nacelle_angle_sin'] = np.cos(np.sin(data['Nacelle_angle']))

data['Nacelle_angle_min_cos'] = np.cos(np.cos(data['Nacelle_angle_min']))
data['Nacelle_angle_min_sin'] = np.cos(np.sin(data['Nacelle_angle_min']))

data['Nacelle_angle_max_cos'] = np.cos(np.cos(data['Nacelle_angle_max']))
data['Nacelle_angle_max_sin'] = np.cos(np.sin(data['Nacelle_angle_max']))

to_remove = ['Nacelle_angle', 'Nacelle_angle_min', 'Nacelle_angle_max']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
data['Pitch_angle_cos'] = np.cos(np.cos(data['Pitch_angle']))
data['Pitch_angle_sin'] = np.cos(np.sin(data['Pitch_angle']))

data['Pitch_angle_min_cos'] = np.cos(np.cos(data['Pitch_angle_min']))
data['Pitch_angle_min_sin'] = np.cos(np.sin(data['Pitch_angle_min']))

data['Pitch_angle_max_cos'] = np.cos(np.cos(data['Pitch_angle_max']))
data['Pitch_angle_max_sin'] = np.cos(np.sin(data['Pitch_angle_max']))

to_remove = ['Pitch_angle', 'Pitch_angle_min', 'Pitch_angle_max']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
data['Absolute_wind_direction_cos'] = np.cos(np.cos(data['Absolute_wind_direction']))
data['Absolute_wind_direction_sin'] = np.cos(np.sin(data['Absolute_wind_direction']))

to_remove = ['Absolute_wind_direction']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

On travaille sur la turbine n°4 :

In [ ]:
data = data[data['MAC_CODE'] == 'WT4']
print('Dimensions du jeu de données :', data.shape)

On observe le lien entre `TARGET` et `Rotor_speed` :

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data['Rotor_speed'], data['TARGET'])
ax.grid()
ax.set_xlabel('Rotor_speed')
ax.set_ylabel('TARGET')
plt.show()

On retire les puissances nulles ou négatives :

In [ ]:
data = data[data['TARGET'] > 0]
print('Dimensions du jeu de données :', data.shape)

On échantillonne à 1/10 pour limiter les calculs dans ce notebook (on fixe la graine du générateur aléatoire pour éventuellement comparer les résultats):

In [ ]:
sample_frac = 1/10
data = data.sample(frac=sample_frac, random_state=1789)
print('Dimensions du jeu de données :', data.shape)

### 1.4. Gestion des variables

In [ ]:
features = list(data.columns.difference(['TARGET', 'MAC_CODE', 'Date_time']))

X = data[features]
y = data['TARGET']

### 1.5. Création d'un échantillon test

On fixe la graine du générateur aléatoire pour éventuellement comparer les résultats.

In [ ]:
test_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion, random_state=1789)

print('Dimensions de X_train_valid :', X_train_valid.shape)
print('Dimensions de X_test        :', X_test.shape)
print('Dimensions de y_train_valid :', y_train_valid.shape)
print('Dimensions de y_test        :', y_test.shape)

### 1.6. Normalisation des covariables

On produit le jeu de données normalisé (centrage-réduction) :

In [ ]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train_valid)

X_train_valid_norm = scaler.transform(X_train_valid)
X_test_norm        = scaler.transform(X_test)

---
## 2. Modélisation
---